In [ ]:
from pathlib import Path

import pandas as pd


PROJECT_ROOT = Path.cwd().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

collisions = pd.read_csv(
    RAW_DATA_DIR / "collisions_2020_2024.csv",
    usecols=[
        "collision_index",
        "collision_year",
        "collision_severity",
        "number_of_vehicles",
        "number_of_casualties",
        "date",
        "day_of_week",
        "time",
        "longitude",
        "latitude",
        "weather_conditions",
        "light_conditions",
        "road_surface_conditions",
        "speed_limit",
        "urban_or_rural_area",
    ],
    low_memory=False,
)

casualties = pd.read_csv(
    RAW_DATA_DIR / "casualties_2020_2024.csv",
    usecols=[
        "collision_index",
        "collision_year",
        "vehicle_reference",
        "casualty_reference",
        "casualty_class",
        "casualty_severity",
        "casualty_type",
        "sex_of_casualty",
        "age_of_casualty",
        "age_band_of_casualty",
    ],
    low_memory=False,
)

vehicles = pd.read_csv(
    RAW_DATA_DIR / "vehicles_2020_2024.csv",
    usecols=[
        "collision_index",
        "collision_year",
        "vehicle_reference",
        "vehicle_type",
        "sex_of_driver",
        "age_of_driver",
        "age_band_of_driver",
    ],
    low_memory=False,
)

print("Collisions:", collisions.shape)
print("Casualties:", casualties.shape)
print("Vehicles:", vehicles.shape)

In [ ]:
casualties["casualty_key"] = (
    casualties["collision_index"].astype(str)
    + "-"
    + casualties["casualty_reference"].astype(str)
)

vehicles["vehicle_key"] = (
    vehicles["collision_index"].astype(str)
    + "-"
    + vehicles["vehicle_reference"].astype(str)
)

print(
    "Unique casualties:",
    f"{casualties['casualty_key'].nunique():,}",
)

print(
    "Unique vehicles:",
    f"{vehicles['vehicle_key'].nunique():,}",
)

In [ ]:
CURRENT_YEAR = 2024
PREVIOUS_YEAR = 2023

collisions_current = collisions[
    collisions["collision_year"] == CURRENT_YEAR
].copy()

casualties_current = casualties[
    casualties["collision_year"] == CURRENT_YEAR
].copy()

vehicles_current = vehicles[
    vehicles["collision_year"] == CURRENT_YEAR
].copy()

total_collisions = collisions_current["collision_index"].nunique()
total_casualties = casualties_current["casualty_key"].nunique()
total_vehicles = vehicles_current["vehicle_key"].nunique()

print(f"Total collisions in {CURRENT_YEAR}: {total_collisions:,}")
print(f"Total casualties in {CURRENT_YEAR}: {total_casualties:,}")
print(f"Total vehicles involved in {CURRENT_YEAR}: {total_vehicles:,}")

In [ ]:
severity_lookup = {
    1: "Fatal",
    2: "Serious",
    3: "Slight",
}

casualties_current["severity_label"] = (
    casualties_current["casualty_severity"]
    .map(severity_lookup)
    .fillna("Unknown")
)

casualties_by_severity = (
    casualties_current.groupby("severity_label")
    .agg(total_casualties=("casualty_key", "nunique"))
    .reset_index()
    .sort_values("total_casualties", ascending=False)
)

casualties_by_severity

In [ ]:
yearly_kpis = (
    collisions.groupby("collision_year")
    .agg(
        total_collisions=("collision_index", "nunique"),
        reported_casualties=("number_of_casualties", "sum"),
    )
    .reset_index()
)

yearly_kpis["collision_change_percent"] = (
    yearly_kpis["total_collisions"].pct_change() * 100
)

yearly_kpis["casualty_change_percent"] = (
    yearly_kpis["reported_casualties"].pct_change() * 100
)

yearly_kpis